# Lorenz-1960 PINN (FYDP-2)

Physics-Informed Neural Network that solves the Lorenz-1960 ODE system.
Hard initial condition (trial solution), residual-only loss, Adam with
polynomial LR decay, adaptive RAR/RAD collocation. Validated against the
locked RK4/SciPy baseline.

Runs on **Kaggle / Colab** (GPU used automatically if available).

## Setup
On **Colab**, clone the repo first (uncomment). On **Kaggle**, add the repo
as a dataset/utility script, then point `REPO_ROOT` at it. Locally, just run
from the repository root.

In [ ]:
# Colab only (uncomment):
# !git clone https://github.com/ihmorol/fydp_workspace.git
# %cd fydp_workspace

import sys, pathlib
root = pathlib.Path.cwd()
if not (root / 'fydp2').exists() and (root.parent / 'fydp2').exists():
    root = root.parent
sys.path.insert(0, str(root))
print('repo root:', root)

In [ ]:
from fydp2.config import Config, reference_trajectory
from fydp2.train import train, save_results, predict, get_device
print('device:', get_device())

## Configure
Baseline architecture is 4x40 tanh. Change any field here without editing
code. For a quick CPU test reduce `epochs` (e.g. `Config(epochs=2000)`);
on a GPU the default is fine.

In [ ]:
cfg = Config()          # e.g. Config(epochs=2000) for a fast CPU run
cfg

## Train

In [ ]:
model, history = train(cfg)
print('final residual loss: %.3e' % history[-1])

## Evaluate vs the locked baseline

In [ ]:
import numpy as np
metrics = save_results(model, history, cfg)   # writes results/fydp2/
t, ys = reference_trajectory(cfg)
pred = predict(model, t)
rel = np.abs(pred[-1] - ys[-1]) / np.abs(ys[-1])
print('final-state relative error:', dict(zip('xyz', rel.round(6))))
metrics

## Plots

In [ ]:
from IPython.display import Image, display
for name in ['convergence.png', 'solution.png', 'error.png']:
    display(Image(f'{cfg.results_dir}/{name}'))